# 安居客二手房信息爬取与 MongoDB 存储

沿用教材中的安居客案例，使用 `requests` 获取网页，使用 XPath 解析房屋信息，并通过 `pymongo` 将数据保存到 MongoDB。

提取字段：房源标题、楼盘名称、地址、房屋户型、建筑面积、所在楼层、建造年代、每平方米单价。

## 1. 安装依赖

如果当前环境尚未安装相关库，运行下面的单元格。

In [1]:
%pip install requests lxml pymongo

   ---------------------------------------- 0.0/829.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/829.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/829.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/829.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/829.1 kB ? eta -:--:--
   ------------------------ ------------- 524.3/829.1 kB 577.4 kB/s eta 0:00:01
   ------------------------ ------------- 524.3/829.1 kB 577.4 kB/s eta 0:00:01
   ------------------------ ------------- 524.3/829.1 kB 577.4 kB/s eta 0:00:01
   ---------------------------------------- 829.1/829.1 kB 533.3 kB/s  0:00:01

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ------------------

In [2]:
import time
import requests
from lxml import etree
from urllib.parse import urljoin
from pymongo import MongoClient
from pprint import pprint

## 2. 数据请求

使用 GET 请求获取网页 HTML。请求头沿用教材中的 User-Agent 设置。

In [3]:
BASE_URL = "https://www.bspider.top/anjuke/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_html(url, timeout=30):
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.encoding = response.apparent_encoding
    response.raise_for_status()
    return response.text

先请求第一页，确认返回的是房源列表页面。

In [4]:
html = get_html(BASE_URL)
print(html[:500])


<!doctype html>
<html lang="en">
<head>
    <title>重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客</title>
    <meta data-n-head="ssr" charset="utf-8">
    <meta data-n-head="ssr" name="viewport"
          content="width=device-width, initial-scale=1, maximum-scale=1.0, minimum-scale=1.0,user-scalable=no,viewport-fit=cover">
    <meta data-n-head="ssr" name="location" content="province=重庆市;city=重庆;coord=106.558106,29.56911">
    <meta data-n-head="ssr" name="mobile-agent" content="format=html5; url=https:/


## 3. 数据解析

每条房源对应一个 `div.property`。通过 XPath 提取标题、楼盘、地址、户型、面积、楼层、建造年代和单价，最终整理成字典列表。

In [5]:
def parser(html):
    doc = etree.HTML(html)
    house_list = []

    for row in doc.xpath("//div[contains(concat(' ', normalize-space(@class), ' '), ' property ')]"):
        title = "".join(row.xpath(".//h3[1]//text()")).strip()
        house = "".join(row.xpath(".//*[contains(@class, 'property-content-info-comm-name')]/text()")).strip()
        address = row.xpath("string(.//*[contains(@class, 'property-content-info-comm-address')])").strip()
        info = [x.strip() for x in row.xpath(".//*[contains(@class, 'property-content-info-attribute')]//p[contains(@class, 'property-content-info-text')]/text()")]
        price = "".join(row.xpath(".//*[contains(@class, 'property-price-average')]/text()")).replace(" ", "").strip()

        house_list.append({
            "title": title,
            "house": house,
            "address": address,
            "struct": info[0] if len(info) > 0 else "",
            "area": info[1] if len(info) > 1 else "",
            "floor": info[3] if len(info) > 3 else "",
            "maketime": info[4] if len(info) > 4 else "",
            "price": price
        })

    return house_list

In [6]:
first_page_data = parser(html)
print("第一页提取到", len(first_page_data), "条房源信息")
pprint(first_page_data[:2])

第一页提取到 60 条房源信息
[{'address': '南岸学府大道二塘路59号',
  'area': '',
  'floor': '',
  'house': '万科金域学府翰林菁华里',
  'maketime': '',
  'price': '16465元/㎡',
  'struct': '',
  'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'},
 {'address': '渝北渝北黄泥磅紫荆路29号',
  'area': '',
  'floor': '',
  'house': '两代一家',
  'maketime': '',
  'price': '14815元/㎡',
  'struct': '',
  'title': '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住'}]


## 4. 分页爬取

教材中“下一页”按钮的 `href` 指向下一页地址，且最后一页不再包含 `next-active`。这里使用循环依次请求各页，最多爬取 10 页。

In [7]:
def crawl(start_url, max_pages=10):
    all_data = []
    current_url = start_url
    visited = set()
    page = 1

    while current_url and current_url not in visited and page <= max_pages:
        visited.add(current_url)
        page_html = html if page == 1 else get_html(current_url)
        page_data = parser(page_html)
        all_data.extend(page_data)
        print(f"第 {page} 页：提取 {len(page_data)} 条")

        doc = etree.HTML(page_html)
        next_link = doc.xpath("//a[contains(concat(' ', normalize-space(@class), ' '), ' next-active ')]/@href")
        if not next_link:
            break

        current_url = urljoin(current_url, next_link[0])
        page += 1
        time.sleep(1)

    return all_data

house_data = crawl(BASE_URL, max_pages=10)
print("总计提取：", len(house_data), "条")

第 1 页：提取 60 条
第 2 页：提取 60 条
第 3 页：提取 60 条
第 4 页：提取 60 条
第 5 页：提取 60 条
第 6 页：提取 60 条
第 7 页：提取 60 条
第 8 页：提取 60 条
第 9 页：提取 60 条
第 10 页：提取 60 条
总计提取： 600 条


## 5. 连接 MongoDB

连接本机 MongoDB 服务，数据库名称为 `anjuke`，集合名称为 `houseinfo`。首次写入数据时，MongoDB 会自动创建数据库和集合。

In [8]:
client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=3000)
client.admin.command("ping")

db = client["anjuke"]
collection = db["houseinfo"]
print("MongoDB 连接成功")

MongoDB 连接成功


## 6. 数据存储

MongoDB 以文档形式保存数据，每条房源信息对应一个文档。

In [9]:
def save_mongodb(collection, data):
    if data:
        result = collection.insert_many(data)
        print("成功存储", len(result.inserted_ids), "条数据")
    else:
        print("没有可存储的数据")

save_mongodb(collection, house_data)

成功存储 600 条数据


## 7. 查看 MongoDB 存储结果

查询集合中的记录数，并查看前 5 条房源文档。

In [10]:
print("集合中的数据总数：", collection.count_documents({}))

for item in collection.find({}, {"_id": 0}).limit(5):
    pprint(item)

集合中的数据总数： 600
{'address': '南岸学府大道二塘路59号',
 'area': '',
 'floor': '',
 'house': '万科金域学府翰林菁华里',
 'maketime': '',
 'price': '16465元/㎡',
 'struct': '',
 'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器'}
{'address': '渝北渝北黄泥磅紫荆路29号',
 'area': '',
 'floor': '',
 'house': '两代一家',
 'maketime': '',
 'price': '14815元/㎡',
 'struct': '',
 'title': '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住'}
{'address': '渝北鸳鸯龙展路59号',
 'area': '',
 'floor': '',
 'house': '大雅云居山',
 'maketime': '',
 'price': '16487元/㎡',
 'struct': '',
 'title': '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售'}
{'address': '渝北中央公园翠丹路4号',
 'area': '',
 'floor': '',
 'house': '中交中央公园悦颂',
 'maketime': '',
 'price': '20660元/㎡',
 'struct': '',
 'title': '中交中央公园 横厅精装大两房 出门都是公园'}
{'address': '南岸弹子石弹子石新街59号',
 'area': '',
 'floor': '',
 'house': '富力现代广场',
 'maketime': '',
 'price': '16969元/㎡',
 'struct': '',
 'title': '富力现代广场 豪装大两房 户型方正 看房方便'}
